# Electricity Load Diagrams 2011–2014

## Постановка задачи

Исследуем периоды наблюдений, совокупную нагрузку, временные закономерности и различия клиентов. Результаты нужны для планирования мощности, поиска пиков и подготовки прогноза на следующие 96 интервалов; здесь выполняется только EDA. Подробная постановка — в [бизнес-задаче проекта](../BUSINESS_TASK.md).

### Единицы измерения и допущения

По [описанию UCI](https://archive.ics.uci.edu/dataset/321/electricityloaddiagrams20112014), столбцы обозначают клиентов или точки учёта, значения — мощность в кВт; энергия за 15 минут равна мощности × 0,25 часа. Временные метки относятся к португальскому времени, а переходы часов имеют особое представление в данных.

Для календарных расчётов исходная метка трактуется как конец интервала и сдвигается на 15 минут назад. Это рабочее допущение: описание источника не подтверждает однозначно начало или конец интервала. В таблицах отдельных событий остаются исходные метки, в суточных профилях используется начало интервала.

## 1. Знакомство с данными

Загружаем исходный файл и проверяем размер, типы столбцов и временной охват. Это задаёт доступный период исследования и число клиентов, из которых складывается будущий прогноз общей нагрузки.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

project_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "src" / "data_load.py").exists()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError("Не удалось найти корень проекта ElectricityLoadDiagrams.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_load import get_data
from src import data_validation as validation
from src import eda_utils as eda
from src import visualizations as viz
from src import eda_reporting as reporting

viz.configure_eda_plot_style()

In [ ]:
data = get_data()

In [ ]:
dataset_overview = validation.get_dataset_overview(data)

В таблице 140 256 временных отметок и 370 числовых столбцов клиентов, все значения имеют тип `float64`. Исходные метки охватывают период с 01.01.2011 00:15 до 01.01.2015 00:00; при принятом сдвиге это интервалы 2011–2014 годов.

## 2. Первичная проверка качества

Проверяем порядок и уникальность времени, шаг наблюдений и явные пропуски. Для прогноза на 15-минутной сетке нарушения этих условий могли бы исказить календарные признаки и лаги.

In [ ]:
time_index_quality = validation.get_time_index_quality(data)

In [ ]:
value_quality = validation.get_value_quality(data)
value_quality.to_frame()

### Просмотр клиентов и нулевых значений

Просматриваем небольшие срезы, а не все 370 столбцов сразу, и рассчитываем долю нулей. Это помогает отделить особенности начала истории от поведения клиента; нулевое показание само по себе не означает ошибку или отсутствие измерения.

In [ ]:
data.iloc[:5, :10]

In [ ]:
sample_clients = data.sample(
    n=5,
    axis="columns",
    random_state=42,
)

sample_clients.head()

In [ ]:
one_day = sample_clients.loc["2011-01-01"]

print(f"Размер дневного среза: {one_day.shape}")
one_day.head()

In [ ]:
zero_share = validation.calculate_zero_share(data)

Индекс упорядочен; дубликатов времени и имён клиентов, отрицательных и нечисловых/бесконечных значений нет; все 140 255 соседних пар разделены 15 минутами. Срез по исходной дате 01.01.2011 содержит 95 строк, поскольку файл начинается в 00:15; после сдвига меток последний интервал дня учитывается по метке 02.01.2011 00:00. У `MT_001` доля нулей за весь файл — 29,05%, но этот показатель смешивает внутренние нули и периоды вне наблюдаемой активности. Отсутствие `NaN` поэтому не завершает проверку качества: далее уточняем периоды клиентов и исследуем нулевые серии.

### Технические особенности календарных интервалов

Проверяем исходные метки 01:00–01:45 в даты перевода часов: техническое устройство файла может создавать ложные минимумы и скачки общей нагрузки. Среднюю мощность четырёх интервалов сопоставляем с теми же метками предыдущего и следующего дня; это локальный ориентир, не поправка на недельную сезонность. На графиках показаны два примера 2014 года, без сдвига исходных меток и без изменения значений.

In [ ]:
clock_change_audit = eda.get_clock_change_audit(data)
display(clock_change_audit[[
    "season", "mean_power_kw", "reference_mean_kw", "window_to_reference",
    "min_nonzero_clients", "all_zero_intervals",
]].round(3))
fig, axes = viz.plot_clock_change_examples(data)
plt.show()

raw_total_power = data.sum(axis=1)
print(f"Минимум общей мощности: {raw_total_power.min():,.2f} кВт; "
      f"исходная метка {raw_total_power.idxmin()}")

30.03.2014 средняя мощность в окне составляет 454,48 кВт — 0,36% локального ориентира 125 161,56 кВт; здесь же находится минимум всего ряда — 342,60 кВт по исходной метке 01:00. Осенние окна 2012–2014 годов выше ориентира в 1,90–2,00 раза, что согласуется с описанным источником особым учётом перевода часов, а не с обычным профилем нагрузки. В 2011 году сопоставимого эффекта не видно; полностью нулевых строк нет ни в одном из восьми окон, поэтому весеннее окно нельзя трактовать как одновременное отсутствие потребления всех клиентов. Эти интервалы отмечены как технически особые, но значения не заменены; правило их обработки потребует отдельного решения при подготовке данных.

## 3. Наблюдаемые периоды клиентов

Для каждого клиента определяем первое и последнее ненулевое показание и охват по месяцам. Нули вне этого диапазона исключаем из индивидуальных характеристик, внутренние нули сохраняем; это операционное определение периода наблюдений, а не подтверждённые даты подключения и отключения. Проверка нужна, чтобы не принять изменение состава клиентов за изменение их потребления.

In [ ]:
data_clean, activity_mask = eda.get_observed_activity(data)

In [ ]:
activity_periods = eda.get_activity_periods(data_clean, activity_mask)
activity_periods[["start", "end", "duration"]].head()

In [ ]:
activity_overview = eda.get_activity_overview(activity_periods, data.index)
reporting.print_activity_overview(activity_overview)
activity_periods.sort_values("start", ascending=False).head(10)

In [ ]:
monthly_active_clients = eda.get_monthly_activity_coverage(activity_mask)
fig, ax = viz.plot_monthly_activity_coverage(
    monthly_active_clients,
    total_clients=data.shape[1],
)
plt.show()

Ненулевые показания есть у всех 370 клиентов; у 212 они начинаются после начала файла, у 3 заканчиваются раньше его конца. Медианная длительность наблюдаемого диапазона с учётом последнего 15-минутного интервала — 1 096 суток; самая поздняя первая метка у `MT_178` — 18.07.2014 00:15. На графике охват заметно увеличивается в начале 2012 года, поэтому сравнение общей нагрузки между годами относится к меняющейся группе. Месячный охват означает наличие наблюдаемого диапазона в месяце, а не обязательное ненулевое потребление в каждом его интервале.

## 4. Совокупная нагрузка на разных временных масштабах

Суммируем мощность клиентов для каждого 15-минутного интервала и строим часовые, дневные, недельные и месячные сводки. Мощность усредняем или берём её максимум, энергию суммируем; эти величины отвечают разным бизнес-вопросам — необходимой мощности и объёму потребления. Первую неделю показываем на двух разрешениях, чтобы проверить, что скрывает часовое усреднение.

In [ ]:
total_load_15min = eda.get_total_load(data_clean)
total_power_15min = total_load_15min["total_power_kw"]
total_energy_15min = total_load_15min["total_energy_kwh"]
total_load_15min.head()

In [ ]:
hourly_summary = eda.summarize_load(total_power_15min, "1h")
hourly_summary.head()

In [ ]:
week_15min = eda.shift_to_interval_start(total_power_15min)
week_15min = week_15min.loc['2011-01-01':'2011-01-07']
week_hourly = hourly_summary.loc['2011-01-01':'2011-01-07', 'mean_power_kw']

fig, axes = viz.plot_load_resolution_comparison(week_15min, week_hourly)
plt.show()

На первой неделе после 1 января повторяются дневные подъёмы и ночные спады; 1 января имеет заметно более низкий дневной уровень. Часовой ряд сохраняет эту форму, но сглаживает короткие колебания, поэтому для предупреждения кратковременных пиков нужен исходный 15-минутный масштаб. Причину отличия 1 января по этому срезу установить нельзя.

### Дневная нагрузка

Сопоставляем среднюю и максимальную мощность с энергией дня и рассматриваем весь период. Дневной масштаб помогает оценить изменения общего уровня, но не заменяет анализ внутрисуточных пиков.

In [ ]:
daily_summary = eda.summarize_load(total_power_15min, "1D")
daily_summary.head()

In [ ]:
fig, ax = viz.plot_daily_mean_load(daily_summary)
plt.show()

Дневной ряд показывает длительные периоды повышенной нагрузки и отдельные резкие провалы, которые месячная сводка сглаживает. По общей кривой нельзя определить клиента или причину провала; индивидуальные скачки и нулевые серии исследуются в разделе 7.

### Недельная сводка

Объединяем интервалы в календарные недели, начинающиеся в понедельник. При сравнении энергии учитываем полноту недели: сумма зависит не только от нагрузки, но и от числа вошедших дней.

In [ ]:
weekly_summary = eda.summarize_load(total_power_15min, "W-MON")
weekly_summary.head()

Строка с меткой 27.12.2010 включает только доступную часть недели в начале файла, а не данные за декабрь. Её 4,52 млн кВт·ч нельзя напрямую сравнивать с 19,97 млн кВт·ч следующей полной недели.

### Месячная нагрузка

Сравниваем месяцы по средней мощности и энергии. Средняя мощность отделяет уровень нагрузки от разной длины месяца, тогда как энергия характеризует фактически учтённый объём.

In [ ]:
monthly_summary = eda.summarize_load(total_power_15min, "MS")
monthly_summary.head()

In [ ]:
fig, ax = viz.plot_monthly_mean_load(monthly_summary)
plt.show()

Средняя мощность в январе 2011 года — 114 598,10 кВт, в мае — 129 100,02 кВт. На графике 2012–2014 годов повторяется повышение нагрузки в середине года, а на границе 2011–2012 годов виден скачок уровня одновременно с увеличением охвата клиентов. Это сезонный рисунок совокупного ряда, но не доказательство роста потребления каждого клиента; количественно отделить эффект состава здесь нельзя.

### Согласованность агрегатов

Проверяем, что суммарная мощность совпадает с суммой клиентов, а энергия сохраняется при переходе к часовому, дневному и месячному масштабу.

In [ ]:
load_checks = eda.validate_load_aggregations(
    raw_data=data,
    total_power=total_power_15min,
    total_energy=total_energy_15min,
    hourly_summary=hourly_summary,
    daily_summary=daily_summary,
    monthly_summary=monthly_summary,
)
print('Проверки суммы мощности и энергии пройдены.')

Арифметические проверки мощности и энергии пройдены. Они подтверждают согласованность расчётов, но не доказывают правильность рабочего допущения о временных метках.

## 5. Временные закономерности

Добавляем календарные признаки и сравниваем нагрузку по часу и дню недели, затем — форму суточного профиля по годам. Это проверка оснований для календарных признаков и суточных/недельных базовых прогнозов; качество самого прогноза на этом этапе не оценивается.

In [ ]:
total_load_15min = eda.add_calendar_features(total_load_15min)
total_load_15min.head()

### Недельный профиль и сравнение лет

Тепловая карта показывает медианную мощность для каждой комбинации дня недели и часа. Для сравнения лет медианные суточные профили делим на их средний уровень: так сопоставляем форму без различий в абсолютном масштабе нагрузки.

In [ ]:
weekday_hour_profile = eda.get_weekday_hour_profile(total_load_15min)
weekly_heatmap = eda.get_weekly_heatmap(weekday_hour_profile)
weekday_hour_profile

In [ ]:
fig, ax = viz.plot_weekly_load_heatmap(weekly_heatmap)
plt.show()

In [ ]:
yearly_normalized_daily_profiles = eda.get_yearly_normalized_daily_profiles(
    total_load_15min
)
year_profile_correlations = yearly_normalized_daily_profiles.corr()

fig, ax = viz.plot_yearly_normalized_daily_profiles(
    yearly_normalized_daily_profiles
)
plt.show()
year_profile_correlations

На всех днях недели нагрузка ниже ночью и выше днём и вечером; различия между часами заметнее различий между днями на общей тепловой карте. Корреляции нормированных суточных профилей 2011–2014 годов лежат в диапазоне 0,9642–0,9972, что подтверждает сходство агрегированной формы. Это не оценка точности прогноза и не проверка стабильности отдельных клиентов; недельные различия далее уточняем сравнением типов дней.

### Суточный профиль: будни и выходные

Сравниваем медианную мощность по 96 слотам, энергию дней и время типичного максимума. Полоса 25–75% на общем профиле отражает разброс наблюдений разных дат в одном слоте, включая изменения уровня между годами, а не доверительный интервал. Будни и выходные определены по дню недели; праздники и производственные календари не учтены.

In [ ]:
daily_load_profile = eda.get_daily_load_profile(total_load_15min)
daily_load_profile.head()

In [ ]:
fig, ax = viz.plot_typical_daily_profile(daily_load_profile)
plt.show()

In [ ]:
day_profile, day_type_summary = eda.get_day_type_profiles(total_load_15min)
largest_day_type_differences = (
    day_profile.reindex(
        day_profile["median_difference_kw"].abs().sort_values(ascending=False).index
    )
    .head(10)
)
display(day_type_summary, largest_day_type_differences)

In [ ]:
fig, ax = viz.plot_day_type_profiles(day_profile)
plt.show()

В сравнении 1 043 будних и 418 выходных дней медианная суточная энергия составляет соответственно 4,698 и 4,626 млн кВт·ч. Самое большое абсолютное различие медианных профилей — в 08:30: выходные ниже будней на 8 961,69 кВт. Типичный максимум приходится на 18:30 в будни и 19:00 в выходные, при близкой общей форме профилей. Различия описательные: состав клиентов и распределение дат по сезонам отдельно не выравнивались.

### Типичные пики и события высокой нагрузки

Сначала ищем максимумы медианного профиля в заранее заданных окнах 06:00–11:45 и 16:00–22:45, затем отдельно рассматриваем редкие высокие нагрузки. Максимум внутри окна не обязательно является самостоятельным локальным пиком; максимум типичного профиля также не равен максимальному наблюдавшемуся значению.

In [ ]:
peak_windows = {
    "Утро (06:00–11:45)": (24, 47),
    "Вечер (16:00–22:45)": (64, 91),
}

typical_power_kw = daily_load_profile["median_power_kw"]
peak_summary = eda.get_window_peak_summary(typical_power_kw, peak_windows)
day_type_median_profiles, day_type_peak_summary = eda.get_day_type_window_peaks(
    total_load_15min,
    peak_windows,
)
display(peak_summary, day_type_peak_summary)

In [ ]:
fig, ax = viz.plot_daily_profile_with_peaks(daily_load_profile, peak_summary)
plt.show()

Максимум вечернего окна — 258 734,34 кВт в 18:30; утреннего — 238 623,06 кВт в 11:45. Утренний максимум находится на правой границе окна, поэтому отдельный утренний пик этим расчётом не подтверждён. Для выходных максимум утреннего окна приходится на 11:15, вечернего — на 19:00; все эти значения относятся к типичному медианному профилю.

### Редкие высокие нагрузки

Выделяем интервалы не ниже 99-го процентиля совокупной мощности и объединяем соседние интервалы в события. Сравниваем мощность, длительность и время максимума событий: это отдельная проверка периодов, наиболее требовательных к доступной мощности.

In [ ]:
high_load_events, high_load_threshold_kw = eda.get_high_load_events(
    total_load_15min,
    quantile=0.99,
)
peak_event_timing = eda.get_peak_event_timing(high_load_events)

print(f"Порог 99-го процентиля: {high_load_threshold_kw:,.2f} кВт")
high_load_events.head(10)

In [ ]:
fig, ax = viz.plot_peak_event_timing(peak_event_timing)
plt.show()

Порог высокой нагрузки, равный 99-му процентилю ряда, — 402 775,46 кВт. Событие с наибольшей мощностью достигло 452 652,60 кВт 09.07.2013 в 17:00 по исходной метке и содержит 40 интервалов выше порога, то есть 10 часов. По диаграмме пики таких событий чаще приходятся на дневные часы с наибольшей частотой около 16 часов, а не на 18:30 типичного профиля. Для планирования мощности поэтому важно различать обычный суточный максимум и редкие высокие нагрузки; сам процентиль ещё не является согласованным бизнес-порогом уведомления.

## 6. Поведение клиентов: вклад и изменчивость

Ранжируем клиентов по энергии за весь доступный период и сравниваем этот вклад с долей в общем максимуме и интервалах высокой нагрузки. Это позволяет выделить клиентов, значимых для планирования мощности, не подменяя пиковую нагрузку суммарным объёмом энергии.

In [ ]:
client_consumption_summary = eda.get_client_energy_summary(data_clean)
client_consumption_summary.head(10)

In [ ]:
client_peak_contributions = eda.get_client_peak_contributions(
    data_clean,
    total_power_15min,
    high_load_quantile=0.99,
)

print(
    "Максимум совокупной нагрузки:",
    client_peak_contributions.attrs["system_peak_timestamp"],
    f"({client_peak_contributions.attrs['system_peak_power_kw']:,.2f} кВт)",
)
client_peak_contributions.head(10)

In [ ]:
fig, axes = viz.plot_client_share_pareto(client_consumption_summary, top_n=20)
plt.show()

`MT_362` и `MT_196` дают 19,23% и 10,61% всей энергии, вместе — 29,84%. В общем максимуме нагрузки вклад `MT_362` выше — 34,07%; в энергии интервалов не ниже 99-го процентиля — 34,57%. Следовательно, его значение для сильных нагрузок больше, чем показывает доля за весь период. Суммарная энергия зависит также от длительности наблюдений, поэтому этот рейтинг не равен рейтингу средней интенсивности потребления.

### Изменчивость суточной энергии

Оцениваем клиентов по медианному абсолютному отклонению суточной энергии, делённому на её медиану (`relative_daily_mad_pct`). Используем только полные наблюдаемые сутки; показатель характеризует междневной разброс, но сам по себе не отделяет сезонность и плановый режим от непредсказуемого поведения.

In [ ]:
client_instability, undefined_instability_clients = eda.get_client_daily_instability(
    data_clean
)
relative_daily_energy_mad_pct = client_instability[
    "relative_daily_mad_pct"
].head(10)

In [ ]:
display(
    client_instability.head(10),
    undefined_instability_clients,
)

In [ ]:
fig, ax = viz.plot_client_instability(
    client_consumption_summary,
    client_instability,
)
plt.show()

Самый высокий относительный MAD у `MT_132` — 96,58%, затем у `MT_093` — 90,27% и `MT_332` — 89,50%. Эти клиенты выделяются по выбранной мере изменчивости, но не являются крупнейшими по суммарной энергии. У `MT_347` медиана суточной энергии равна нулю, а 56,28% из 1 082 полных дней нулевые: относительный показатель не определён, и его нельзя трактовать как стабильное нулевое потребление.

## 7. Скачки и продолжительные нулевые серии

Рассматриваем 99-й процентиль абсолютных изменений между соседними интервалами и его отношение к медианной ненулевой мощности клиента. Абсолютный масштаб важен для общей нагрузки, относительный — для необычности изменения у конкретного клиента; это не процент роста относительно предыдущего интервала и не максимальный скачок. Отдельные события проверяем по соседним показаниям, не объявляя их ошибками автоматически.

In [ ]:
change_kw, spike_ranking = eda.get_spike_ranking(data_clean)
display(
    spike_ranking.nlargest(10, "p99_abs_change_kw"),
    spike_ranking.nlargest(10, "p99_relative_change_pct"),
)

### Проверка отдельных скачков

Для пяти лидеров абсолютного рейтинга рассматриваем по два крупнейших события. Признаки касания нуля и совпадения с потенциальным окном перевода часов помогают выбрать случаи для проверки, но не устанавливают причину скачка.

In [ ]:
clock_change_windows = eda.get_clock_change_windows(data_clean.index)
spike_clients = spike_ranking.nlargest(5, "p99_abs_change_kw").index
spike_events = eda.get_top_spike_events(
    data_clean,
    change_kw,
    clients=spike_clients,
    events_per_client=2,
    clock_change_windows=clock_change_windows,
)
spike_events.round(1)

In [ ]:
fig, ax = viz.plot_relative_spike_ranking(spike_ranking, top_n=10)
plt.show()

У `MT_362` 99-й процентиль абсолютного изменения равен 30 800 кВт. Относительный рейтинг возглавляет `MT_093`: 3 314,29% при медианной ненулевой мощности всего 3,55 кВт, поэтому большой процент не означает большой вклад в общий пик. Рейтинг определяет приоритет проверки, но сам по себе не доказывает ошибку измерения.

### Локальная проверка скачков

Рассматриваем крупнейшее событие `MT_362` и два крупнейших события `MT_279` в окружении ±2 часов. Это позволяет отличить резкое изменение с продолжением от краткого провала и уточнить кандидатов для проверки перед прогнозированием; выборка целевая, не оценка частоты всех аномалий.

In [ ]:
spike_cases = pd.concat([
    spike_events.xs("MT_362", level="client", drop_level=False).head(1),
    spike_events.xs("MT_279", level="client", drop_level=False).head(2),
])
display(spike_cases[["previous_kw", "current_kw", "next_kw", "jump_kw"]].round(1))
fig, axes = viz.plot_spike_case_contexts(data_clean, spike_cases)
plt.show()

У `MT_362` 21.09.2012 в 09:30 мощность меняется 46 000 → 156 000 → 92 400 кВт: после экстремума она не возвращается сразу к прежнему уровню. У `MT_279` 11 и 18.11.2014 в 16:30 повторяется однократный провал 12 863,3 → 104,2 → 12 445,8 кВт; оба события происходят во вторник с интервалом в неделю. Два примера показывают повторение, но не устанавливают недельную закономерность или причину провала. Сценарии различаются, поэтому автоматическое удаление всех скачков одним правилом здесь не обосновано.

### Нулевые эпизоды внутри наблюдаемого периода

Выделяем непрерывные серии нулей внутри диапазона клиента, затем сравниваем максимальную длительность и число эпизодов не короче часа. Длительность равна числу нулевых интервалов × 15 минут, а не просто разности первой и последней меток. Совпадение с потенциальным окном перевода часов отмечается отдельно и не означает доказанную связь.

In [ ]:
systemwide_zero_times = data.index[data.eq(0).all(axis=1)]
all_zero_periods = eda.get_zero_periods(
    data_clean,
    systemwide_zero_times=systemwide_zero_times,
    clock_change_windows=clock_change_windows,
)
display(all_zero_periods.head(10))

In [ ]:
zero_periods_summary = eda.get_zero_periods_summary(
    all_zero_periods,
    clients=data_clean.columns,
)
zero_periods_summary.head(10)

In [ ]:
fig, ax = viz.plot_longest_zero_periods(zero_periods_summary, top_n=10)
plt.show()

Самые длинные внутренние нулевые серии: `MT_347` — 10 761 час, `MT_015` — 2 640 часов, `MT_131` — 1 230,5 часа. У `MT_131` также 1 311 эпизодов длительностью не меньше часа, то есть важна не только длина, но и повторяемость нулей. Сводка выявляет случаи для проверки статуса клиента и качества измерений, но не отличает реальное отсутствие потребления от особенностей учёта. Без дополнительной информации нет основания автоматически заменять все нули пропусками или интерполировать длинные серии.

### Контекст самой длинной нулевой серии

Для `MT_347` показываем границы самого длинного внутреннего нулевого периода и среднюю суточную мощность с запасом по семь дней с обеих сторон. Проверяем, что серия лежит между наблюдаемыми ненулевыми показаниями; график использует ту же рабочую привязку к началу 15-минутного интервала, что и остальные суточные агрегации.

In [ ]:
zero_case = all_zero_periods.loc[
    all_zero_periods["client"].eq("MT_347")
].nlargest(1, "intervals").iloc[0]
display(zero_case[["client", "start", "last_zero", "intervals", "duration"]].to_frame("value"))
fig, ax = viz.plot_zero_period_context(data_clean, zero_case)
plt.show()

У `MT_347` серия содержит 43 044 последовательных нулевых интервала: от исходной метки 07.12.2012 00:30 до 28.02.2014 09:15, суммарно 10 761 час. До и после неё наблюдаются ненулевые значения — это внутренняя серия, а не только нулевой участок до первой активности клиента. Без сведений о статусе клиента и работе учёта нельзя определить причину нулей или обоснованно восстановить потребление на этом промежутке.

### Концентрация энергии перед сегментацией

Делим клиентов на четыре почти равные по численности группы по суммарной энергии, чтобы оценить концентрацию нагрузки по объёму. Это квартильные группы, а не кластеры похожих суточных профилей; доли относятся ко всей энергии 370 клиентов за весь файл.

In [ ]:
client_energy_kwh, group_summary = eda.get_energy_groups(
    client_consumption_summary
)
group_summary

В верхней группе Q4 находятся 93 из 370 клиентов, на которых приходится 87,09% энергии; у остальных трёх групп вместе — 12,91%. Концентрация объёма подтверждена, но не определяет сходство формы профилей или вклад квартиля в конкретный пиковый интервал.

## 8. Типичные профили клиентов

Группируем клиентов по форме суточной нагрузки, чтобы описать различия режимов потребления, а не только объёмов. Признаки — 96 медианных 15-минутных слотов внутри наблюдаемого периода, включая внутренние нули; будни, выходные и сезоны пока объединены. Номера кластеров условны и не определяют отрасль, оборудование или причины пиков.

Методы: [KMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html), [silhouette](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html), [Adjusted Rand Index](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html).

In [ ]:
from IPython.display import Markdown, display
from src import profile_clustering as clustering
from src import cluster_visualizations as cluster_viz

MIN_COMPLETE_DAYS = 30
K_VALUES = range(2, 9)
CLUSTER_SEEDS = (42, 17, 73)
SELECTED_K = 4
CLIENTS_PER_CLUSTER = 3

### Построение и нормирование профилей

Медианные значения по времени суток делим на среднее профиля клиента, чтобы убрать абсолютный масштаб; среднее каждой нормированной строки равно 1. Нулевой профиль нормировать нельзя, но он не доказывает отсутствия потребления в исходном ряду. Минимум 30 полных наблюдаемых суток — рабочий порог достаточности истории, его чувствительность проверяется ниже.

In [ ]:
from src.profile_clustering import get_client_typical_profiles

In [ ]:
client_profiles = get_client_typical_profiles(data_clean)

print(client_profiles.shape)
client_profiles.head()

In [ ]:
profile_scale, normalized_profiles = clustering.normalize_client_profiles(client_profiles)
normalized_profiles.head()

### Качество признаков и исключения

Проверяем полноту слотов, допустимость значений и объём истории, сохраняем причины исключения. Это ограничивает выводы кластеризации теми клиентами, чьи выбранные признаки допускают сравнение.

In [ ]:
profile_quality, excluded_profiles, X = clustering.prepare_clustering_features(
    data_clean, client_profiles, normalized_profiles, min_complete_days=MIN_COMPLETE_DAYS,
)

print(f"Исходных клиентов: {len(client_profiles)}")
print(f"После исходной нормализации: {len(normalized_profiles)}")
print(f"В кластеризации: {len(X)}; исключено: {len(excluded_profiles)}")
display(excluded_profiles)
display(profile_quality[[
    "complete_days", "min_slot_observations", "profile_mean_kw",
]].describe().round(2))

threshold_sensitivity = clustering.get_history_threshold_sensitivity(profile_quality)
display(threshold_sensitivity)
for message in reporting.format_profile_exclusions(excluded_profiles):
    display(Markdown(message))

Пороги 0, 7, 30 и 90 полных дней оставляют одинаковые 369 клиентов; минимальная доступная история по проверке — 167 полных дней. `MT_347` исключён из-за нулевого медианного профиля во всех 96 слотах, хотя в исходном ряду у него 32 832 ненулевых интервала. Ограничение связано с выбранным представлением профиля, а не с отсутствием потребления.

### Выбор числа кластеров

Для k от 2 до 8 сравниваем силуэт, размеры групп и согласованность разбиений при трёх seed; KMeans использует 20 инициализаций. Силуэт характеризует разделимость выбранных профилей, ARI — согласованность запусков на одной матрице, а не устойчивость во времени.

In [ ]:
k_comparison, candidate_models, candidate_labels = clustering.compare_cluster_counts(
    X, k_values=K_VALUES, seeds=CLUSTER_SEEDS,
)
display(k_comparison.round(4))
best_silhouette_k = int(k_comparison["silhouette_mean"].idxmax())

fig, axes = cluster_viz.plot_cluster_count_comparison(k_comparison)
plt.show()
display(Markdown(reporting.format_k_comparison(
    k_comparison, candidate_labels, best_silhouette_k,
)))

### Сравнение форм при k = 2, 3 и 4

При k=2 средний силуэт максимален (0,8609), но разбиение выделяет всего 4 клиента против 365 остальных. Начиная с k=5 в проверенных исходных запусках появляются одиночные группы. Сравниваем профили при k=2, 3 и 4 на общей вертикальной шкале: линии — медианы клиентов группы, не центры KMeans.

In [ ]:
candidate_ks = [k for k in (2, 3, 4) if k in candidate_labels]
fig, axes = cluster_viz.plot_candidate_cluster_profiles(X, candidate_labels, candidate_ks)
plt.show()

### Рабочее разбиение на четыре группы

k=4 выбран для более подробного описания форм среди основных клиентов: в этом запуске нет одиночных групп, а минимум составляет 4 клиента. Это описательное решение, не максимум силуэта и не доказательство существования четырёх естественных типов. Согласованность seed далее оценивается отдельно от стабильности профилей между годами.

In [ ]:
if SELECTED_K not in k_comparison.index:
    raise ValueError("SELECTED_K должен присутствовать в таблице сравнения.")

model, labels = clustering.fit_profile_clusters(X, SELECTED_K, random_state=CLUSTER_SEEDS[0])
cluster_summary, cluster_members, cluster_profiles, cluster_q25, cluster_q75 = (
    clustering.summarize_profile_clusters(X, labels, client_consumption_summary)
)
cluster_sizes = labels.value_counts().sort_index()
representative_clients = clustering.cluster_representatives(
    cluster_members, per_cluster=CLIENTS_PER_CLUSTER,
)
display(cluster_sizes.rename("clients").to_frame())
display(k_comparison.loc[[SELECTED_K]].round(4))

selected_quality = k_comparison.loc[SELECTED_K]
display(Markdown(reporting.format_selected_clustering(
    SELECTED_K, selected_quality, cluster_members,
)))

### Профили и внутригрупповой разброс

Показываем медиану и диапазон 25–75% по клиентам в каждом слоте. Полоса отражает различия типичных профилей внутри группы, не междневную изменчивость и не доверительный интервал. У панелей разные вертикальные шкалы; среднее групповой покомпонентной медианы не обязано равняться 1, даже после нормирования каждого клиента.

In [ ]:
fig, axes = cluster_viz.plot_cluster_profiles(
    cluster_profiles, cluster_q25, cluster_q75, cluster_sizes,
)
plt.show()

shape_columns = [
    "clients", "peak_slot", "peak_normalized", "trough_slot",
    "trough_normalized", "night_level", "morning_level",
    "day_level", "evening_level", "mean_slot_iqr",
]
display(cluster_summary[shape_columns].round(3))
display(Markdown(reporting.format_profile_window_note()))

Кластеры 0 и 2 отличаются глубиной ночного спада: минимум их медианных нормированных профилей — 0,278 и 0,566; у кластера 3 выражен вечерний максимум в 20:00. Маленький кластер 1 включает 4 клиентов с наибольшим средним уровнем в ночном окне и нулевой медианой в дневном и вечернем окнах. Это различия типичных форм, не утверждение о нулевом потреблении каждого клиента во все соответствующие дни.

### Индивидуальные примеры и слабые назначения

Для групп до 12 клиентов показываем всех, для остальных — три ближайших к медиане профиля и наиболее удалённый. Это диагностическая выборка, не случайное представление группы; красный пунктир обозначает наиболее удалённый показанный профиль.

In [ ]:
display(representative_clients.join(cluster_members[[
    "distance_to_median_rmse", "silhouette",
]]).round(4))

fig, axes = cluster_viz.plot_cluster_examples(
    X, cluster_profiles, cluster_members, representative_clients,
)
plt.show()

Профиль `MT_093` заметно отличается от медианы своей группы (RMSE 1,2360), хотя его силуэт остаётся положительным (0,0325). Поэтому одной групповой медианы недостаточно для характеристики всех клиентов.

### Границы кластеров

Рассматриваем индивидуальные силуэты и наиболее слабые назначения. Отрицательное значение указывает на слабое соответствие кластеру, но не доказывает аномалию самого потребления.

In [ ]:
fig, ax = cluster_viz.plot_cluster_silhouettes(cluster_members)
plt.show()
display(cluster_members.nsmallest(10, "silhouette").round(4))

Отрицательный силуэт имеют 9 клиентов: 7 в кластере 3 и 2 в кластере 0. Средний силуэт кластера 3 (0,259) ниже, чем у остальных групп; его назначения требуют большей осторожности.

### Вклад групп в энергию

Дополняем сегментацию по форме числом клиентов и фактической энергией. Доли показываем относительно всех исходных клиентов и только включённых; энергия за весь файл зависит также от длительности наблюдений и не равна вкладу в конкретный пик.

In [ ]:
business_columns = [
    "clients", "client_share_pct", "total_energy_kwh",
    "energy_share_all_pct", "energy_share_included_pct",
    "median_client_mean_power_kw", "silhouette_mean",
    "negative_silhouette_clients", "median_distance_rmse",
]
display(cluster_summary[business_columns].round(3))

all_energy_kwh, included_energy_kwh, excluded_energy_kwh, energy_coverage = (
    clustering.get_clustering_energy_coverage(
        client_consumption_summary, cluster_members, excluded_profiles, X,
    )
)
display(energy_coverage.round(3))
fig, axes = cluster_viz.plot_cluster_energy_summary(cluster_summary)
plt.show()

Кластер 2 включает 218 клиентов и даёт 57,078% всей энергии; кластер 0 — 34 клиента и 29,061%. Ночной кластер из 4 клиентов даёт лишь 0,066%, то есть необычная форма не означает большой вклад в общую нагрузку. Кластеризованные клиенты покрывают 99,999% энергии, исключённый `MT_347` — 0,001%.

### Итоговая таблица и согласованность

Объединяем назначения и показатели качества для всех исходных клиентов, затем проверяем состав, нормирование и сохранение энергии. У исключённого клиента нет метки кластера: исключение не является отдельным найденным типом потребления.

In [ ]:
client_cluster_table = clustering.validate_clustering_result(
    profile_quality, cluster_members, client_profiles, X,
    cluster_sizes, cluster_summary, all_energy_kwh,
    included_energy_kwh, excluded_energy_kwh,
)
display(client_cluster_table.head(10))
print("Проверки состава клиентов, нормирования и энергии пройдены.")

### Результат сегментации и ограничения

Проверки состава клиентов, нормирования и энергии пройдены; ниже сохранена числовая сводка разбиения. Медианы и нормирование скрывают абсолютный масштаб, редкие скачки и междневной разброс, а объединение всех дат не проверяет устойчивость групп между сезонами и годами. Эти кластеры описывают текущие признаки; их полезность для прогноза совокупной нагрузки пока не измерена.

In [ ]:
display(Markdown(reporting.format_cluster_takeaways(
    X, client_profiles, excluded_profiles, energy_coverage,
    selected_quality, cluster_summary, SELECTED_K,
)))

## Основные выводы EDA

- История содержит 370 клиентов и 140 256 отметок с регулярным шагом 15 минут, без явных `NaN` и дубликатов. У 212 клиентов ненулевые показания появляются позже начала файла: общий ряд отражает меняющийся состав, что нужно учитывать при временной проверке будущей модели.
- Суточный рисунок сохраняет сходную форму между годами (корреляции нормированных профилей 0,9642–0,9972), при этом месячный уровень меняется. Это основание проверять календарные признаки и суточные/недельные базовые прогнозы, но не доказательство их будущей точности.
- Типичный максимум — 18:30 и 258 734,34 кВт; в выходные максимум профиля смещается к 19:00. Наблюдавшийся максимум гораздо выше — 452 652,60 кВт: качество прогноза высоких нагрузок следует оценивать отдельно от воспроизведения среднего суточного рисунка.
- 93 клиента дают 87,09% энергии; у `MT_362` доля энергии — 19,23%, а вклад в общий максимум — 34,07%. Крупнейшие потребители требуют отдельного внимания при проверке пиков; концентрация объёма и концентрация пиков не тождественны.
- Междневная изменчивость и нулевые серии неоднородны: относительный MAD `MT_132` — 96,58%, самая длинная нулевая серия `MT_347` — 10 761 час. Эти результаты задают приоритеты проверки, но без дополнительных сведений не обосновывают удаление скачков или заполнение всех нулей.
- Минимум общей мощности 342,60 кВт относится к весеннему техническому окну 30.03.2014, а осенние окна 2012–2014 годов выше соседних дат в 1,90–2,00 раза. Такие интервалы нельзя использовать как обычные наблюдения о минимальной и пиковой нагрузке без оговорки об устройстве файла.
- Кластеризация охватывает 369 клиентов; при k=4 крупнейшая группа включает 218 клиентов и 57,08% энергии. Разбиение согласовано между проверенными seed (ARI 1,0000), но имеет 9 отрицательных силуэтов и не проверено на устойчивость во времени; это описательная сегментация, не готовое основание для прогнозной модели.

Рабочая трактовка меток как конца интервала сохранена; её документальное подтверждение остаётся ограничением. Календарные окна и выбранные аномалии проверены локально, но значения не исправлялись: при подготовке данных для модели потребуется явно выбрать и проверить правила их обработки. Причины пиков, экономический эффект, тарифные решения и режимы оборудования по этим данным не устанавливаются; точность суточного прогноза будет проверяться только на последующих временных отрезках.